# Vaidya — Phase 2: QLoRA Training (Lightning.ai)

**GPU:** A10G 24GB — native BF16  
**Expected runtime:** ~2.5–3 hrs (full epoch)

Assumes the full Vaidya repo is already cloned in `/teamspace/studios/this_studio/`.

## Step 1 — Install packages

In [ ]:
!pip install -q \
    transformers==4.47.0 \
    peft==0.14.0 \
    trl==0.15.2 \
    accelerate==0.34.2 \
    datasets==3.2.0 \
    mlflow==2.19.0 \
    boto3 \
    sentencepiece \
    "fsspec[http]==2024.9.0" \
    "packaging==24.2"

# Force-reinstall bitsandbytes so it picks up the CUDA library, not the CPU fallback
!pip install -q bitsandbytes --upgrade --force-reinstall

# Flash Attention 2 — A100 supports it natively; set CUDA_HOME so the build finds nvcc
!CUDA_HOME=/usr/local/cuda pip install flash-attn --no-build-isolation -q 2>/dev/null || \
    echo "flash-attn build failed — using eager attention"

print('✅ Install done')

## Step 2 — Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU — attach an A10G in Studio settings.'

gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc   = torch.cuda.get_device_capability()

print(f'GPU  : {gpu}')
print(f'VRAM : {vram:.1f} GB')
print(f'CC   : {cc[0]}.{cc[1]}')
print(f'BF16 native : {cc[0] >= 8}')
print(f'Flash Attn  : ', end='')
try:
    import flash_attn; print(f'✅ {flash_attn.__version__}')
except ImportError:
    print('not installed (training still works)')

## Step 3 — Set working directory & verify scripts

In [ ]:
import os, importlib.util

# train.py lives one level up in layer2_finetune/
work_dir = '/teamspace/studios/this_studio/vaidya/layer2_finetune'
os.chdir(work_dir)
print(f'Working dir: {os.getcwd()}')

for fname, modname in [('train.py', 'train_mod'), ('qlora_config.py', 'qlora_config')]:
    spec = importlib.util.spec_from_file_location(modname, f'{work_dir}/{fname}')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    print(f'{fname}  →  version {getattr(mod, "__version__", "MISSING")}')

## Step 4 — AWS credentials

In [ ]:
import os, boto3

# ── Fill in your credentials ────────────────────────────────
os.environ['AWS_ACCESS_KEY_ID']     = 'YOUR_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_ACCESS_KEY'
os.environ['AWS_DEFAULT_REGION']    = 'ap-south-1'
# ────────────────────────────────────────────────────────────

s3 = boto3.client('s3')
buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
print('Buckets:', buckets)
assert 'vaidya-artifacts' in buckets, '❌ vaidya-artifacts not found — check credentials'

## Step 5 — Pull data from S3

In [ ]:
import boto3, os, pandas as pd

os.makedirs('./data', exist_ok=True)
s3 = boto3.client('s3')

for page in s3.get_paginator('list_objects_v2').paginate(Bucket='vaidya-artifacts', Prefix='data/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith('.parquet'):
            fname = key.split('/')[-1]
            local = f'./data/{fname}'
            if not os.path.exists(local):
                print(f'Downloading {key}...')
                s3.download_file('vaidya-artifacts', key, local)
            else:
                print(f'Already exists: {local}')

for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'./data/{split}.parquet')
    print(f'{split}: {len(df):,} rows')

## Step 6 — Pre-flight check

In [ ]:
import importlib

required = ['transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets', 'mlflow', 'boto3']
all_ok = True
for pkg in required:
    try:
        m = importlib.import_module(pkg)
        print(f'✅  {pkg} {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'❌  {pkg}  — {e}')
        all_ok = False

if not all_ok:
    print('\n⚠️  Re-run Step 1 and restart the kernel.')
else:
    print('\n✅ Environment ready — proceed to training.')

## Step 7 — Run training

In [ ]:
!python train.py --help

In [ ]:
# ⚡ FULL TRAINING RUN — A10G ~2.5–3 hrs
!python train.py --run-name run1

## Step 8 — Verify checkpoint on S3

In [ ]:
import boto3

s3 = boto3.client('s3')

print('=== Checkpoint files ===')
resp = s3.list_objects_v2(Bucket='vaidya-artifacts', Prefix='checkpoints/run1/')
for obj in resp.get('Contents', []):
    print(f"  {obj['Key']}  ({obj['Size']/1e6:.1f} MB)")

print()
print('=== MLflow runs ===')
resp2 = s3.list_objects_v2(Bucket='vaidya-artifacts', Prefix='mlflow/')
for obj in resp2.get('Contents', []):
    print(f"  {obj['Key']}")

## Step 9 — Quick loss check

In [ ]:
import json, glob

states = glob.glob('./checkpoints/run1/checkpoint-*/trainer_state.json')
if not states:
    states = glob.glob('./checkpoints/run1/trainer_state.json')

if states:
    with open(sorted(states)[-1]) as f:
        state = json.load(f)
    log_history = state.get('log_history', [])
    train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e]
    if train_losses:
        print(f'Train loss — first: {train_losses[0][1]:.4f}  last: {train_losses[-1][1]:.4f}')
        print('✅ Loss decreased.' if train_losses[-1][1] < train_losses[0][1] else '⚠️ Loss did not decrease.')
else:
    print('No trainer_state.json — check ./checkpoints/run1/')

## Gate check — before Phase 3

- ✅ Training loss decreased
- ✅ Checkpoint at `s3://vaidya-artifacts/checkpoints/run1/`
- ✅ MLflow runs at `s3://vaidya-artifacts/mlflow/`

**Next:** Phase 3 — accuracy eval (base Mistral-7B vs fine-tuned)